# 토큰화(tokenization)

- 문장을 토큰 시퀀스로 나누는 과정

- 수행 대상에 따라 문자, 단어, 서브워드 등의 방법이 있음

- 트랜스포머 모델은 토큰 시퀀스를 입력받으므로 문장에 토큰화를 수행해줘야함

- 토큰화를 수행하는 프로그램을 토크나이저라고 함
    - 한국어 형태소 분석기는 토큰화 뿐만 아니라 품사 부착까지 수행

## 단어 단위 토큰화

- 가장 쉽게 단어 단위 토큰화를 수행하는 방법은 공백으로 분리하는 것
    - 예) 어제 카페 갔었어 -> [어제, 카페, 갔었어]
 
- 공백으로 분리하면 별도로 토크나이저를 쓰지 않아도 된다는 장점이 있지만, 한국어 데이터는 어휘 집합의 크기가 매우 커질 수 있음
    - 갔었어, 갔었는데요 처럼 표현이 살짝만 바뀌어도 모든 경우의 수가 어휘 집합에 포함되어야 하기 때문
 
- 한국어 형태소 분석기를 사용하면 어휘 집합의 크기가 커지는 것을 조금 완화할 수 있음
    - 예) 어제 카페 갔었어 -> [어제, 카페, 갔었, 어]
    - 어제 카페 갔었는데요 -> [어제, 카페, 갔었, 는데요]

- 어휘 집합의 크기가 지나치게 커지면 모델 학습이 어려워질 수 있음

## 문자 단위 토큰화

- 한글 위주로 표현된 데이터로 언어 모델을 만든다면
    - 한글로 표현할 수 있는 글자의 조합 수는 11,172개이므로 알파벳과 숫자, 기호를 합쳐도 어휘 사전의 크기는 1만 5천개를 넘지 않음
 
- 해당 언어의 모든 문자를 어휘 집합에 포함하므로 미등록 토큰 문제로부터 자유로움

- 각 문자 토큰이 의미 있는 단위가 되기 어렵다는 것이 단점
    - 예) 어제 카페 갔었어 -> [어, 제, 카, 페, 갔, 었, 어]
 
- 또한 단어 단위 토큰화에 비해 토큰 시퀀스의 길이가 상대적으로 길어짐
    - 언어 모델에 입력할 토큰 시퀀스가 길면 모델이 해당 문장을 학습하기가 어려워지고 성능이 떨어질 수 있음

## 서브워드 단위 토큰화

- 단어와 문자 단위 토큰화의 중간 형태
    - 어휘 집합 크기가 지나치게 커지지 않으면서 미등록 토큰 문제를 피하고, 분석된 토큰 시퀀스가 너무 길어지지 않게 하기 위한 방법
 
- 대표적인 서브워드 단위 토큰화 기법으로 바이트 페어 인코딩이 있음

### 바이트 페어 인코딩(Byte Pair Encoding;BPE)

- 정보 압축 알고리즘으로 제안되었지만 자연어 처리 모델에 널리 쓰이게 된 토큰화 기법

- GPT 계열 모델은 BPE 기법으로 토큰화를 수행하며, BERT 모델은 BPE와 유사한 워드피드(wordpiece)를 토크나이저로 사용

- 1994년 제안된 정보 압축 알고리즘으로, 데이터에서 가장 많이 등장한 문자열을 병합해서 데이터를 압축하는 기법

- 예시
    - 데이터 : aaabdaaabac
 
    - 데이터에 등장한 글자(a, b, c, d)를 초기 사전으로 구성하며, 연속된 두 글자를 한 글자로 병합
        - aa 가 가장 많이 나타났으므로 이를 Z로 병합
     
    - ZabdZabac
        - ab 가 가장 많이 나타났으므로 이를 Y로 병합
     
    - ZYdZYac
        - ZY를 X로 병합
            - 이미 병합된 문자열 역시 한번 더 병합할 수 있음
         
    - XdXac
 
    - BPE 수행 이전에는 데이터를 표현하기 위한 사전 크기가 4개(a, b, c, d)였지만 수행 이후엔 사전 크기가 7개(a, b, c, d, Z, Y, X)로 늘었지만, 데이터 길이는 11에서 5로 줄었음
 
- BPE는 사전의 크기를 지나치게 늘리지 않으면서 데이터의 길이를 효율적으로 압축하는 알고리즘

- 분석 대상 언어에 대한 지식이 필요 없으며 말뭉치에서 자주 나타나는 문자열(서브워드)을 토큰으로 분석

#### BPE 어휘 집합 구축

- 말뭉치의 모든 문장을 공백으로 나눠 줌
    - 사전 토큰화(pre-tokenize)
 
- 사전 토큰화 결과 예시
    - hug : 10
    - pug : 5
    - pun : 12
    - bun : 4
    - hugs : 5

    - 초기 어휘 집합
        - [b, g, h, n, p, s, u]

- 초기 어휘 집합 기준으로 2개씩(바이그램;bigram) 묶어서 빈도표 작성
    - b, u : 4
    - g, s : 5
    - h, u : 15
    - p, u : 17
    - u, g : 20
    - u, n : 16

- 위에서 가장 많이 등장한 바이그램 쌍은 u, g 이므로 u와 g를 합친 ug를 어휘 집합에 추가
    - [b, g, h, n, p, s, u, ug]

- 새로운 어휘 집합으로 빈도표 작성
    - h, ug : 10
    - p, ug : 5
    - p, u, n : 12
    - b, u, n : 4
    - h, ug, s : 5

- 바이그램 쌍 빈도로 나열
    - b, u : 4
    - h, ug : 15
    - p, u : 12
    - p, ug : 5
    - u, n : 16
    - ug, s : 5

- 이번에 가장 많이 등장한 바이그램 쌍은 u, n 으로 총 16회
    - 따라서 u와 n을 합친 un을 어휘 집합에 추가
    - [b, g, h, n, p, s, u, ug, un]

- BPE 어휘 집합은 사용자가 정한 크기가 될 때까지 이러한 과정을 반복해서 수행
    - 만일 어휘 집합 크기를 9개로 정해 놓았다면 지금 어휘가 9개가 되었으므로 BPE 어휘 집합 구축 절차를 마침

#### BPE 토큰화

- 어휘 집합과 병합 우선순위가 있으면 토큰화를 수행할 수 있음
    - 병합 우선순위
        - u g
        - u n
 
- "pug bug mug" 라는 문장을 토큰화한다면
    - 사전 토큰화 수행
        - pug, bug, mug
     
- 분리된 토큰을 가지고 각각 BPE 토큰화를 수행
    - pug를 문자 단위로 분리
        - p, u, g
     
    - 병합 우선순위를 부여
        - p, u -> 우선순위 없음
        - u, g -> 1순위

    - u와 g를 합침
        - p, ug
     
    - 다시 우선순위 부여
        - p, ug -> 우선순위 없음
     
    - 우선순위 없으므로 병합 종료
 
    - p와 ug가 각각 어휘 집합에 있는지 검사
        - 토큰화 최종 결과는 [p, ug]

- mug를 토큰화 한다면
    - 문자 단위로 분리
        - m, u, g
     
    - 병합 우선순위 부여하여 합침
        - m, ug
     
    - ug는 어휘집합에 있지만, m은 어휘 집합에 없으므로 최종 토큰화는
        - \<unk>, ug
            - unk는 미등록 토큰을 의미
         
- pug bug mug 문장의 BPE 토큰화 결과
    - p, ug, b, ug, \<unk>, ug

### 워드피스(wordpiece)

- 말뭉치에서 자주 등장한 문자열을 토큰으로 인식한다는 점에서 BPE와 본질적으로 유사
    - 다만 어휘 집합을 구축할 때 문자열을 병합하는 기준이 다름
    - 병합했을 때 말뭉치의 우도(likelihood)를 가장 높이는 쌍을 병합
        - 병합 기준 = (ab의 빈도수) / (a의 빈도수 * b의 빈도수)
     
    - 즉, 워드피스에서는 병합 후보에 오른 쌍을 미리 병합해보고 잃는 것과 가치 등을 판단한 후에 병합

# 어휘 집합 구축

In [1]:
from Korpora import Korpora
import os
from tokenizers import ByteLevelBPETokenizer, BertWordPieceTokenizer
from transformers import GPT2Tokenizer, BertTokenizer

In [2]:
# NSMC 다운로드
nsmc = Korpora.load("nsmc", force_download = True)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 24.7MB/s]                                                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 24.1MB/s]                                                             


In [3]:
def write_lines(path, lines):
    with open(path, "w", encoding = "utf-8") as f:
        for line in lines:
            f.write(f"{line}\n")

In [4]:
write_lines("./data/nsmc/train.txt", nsmc.train.get_all_texts())
write_lines("./data/nsmc/test.txt", nsmc.test.get_all_texts())

In [56]:
# BPE 어휘 집합 구축
bytebpe_tokenizer = ByteLevelBPETokenizer()

In [57]:
bytebpe_tokenizer.train(
    files = ["./data/nsmc/train.txt", "./data/nsmc/test.txt"], # 학습 말뭉치를 리스트 형태로 입력
    vocab_size = 10000, # 어휘 집합 크기 조절
    special_tokens = ["[PAD]"] # 특수 토큰 추가
)

In [58]:
bytebpe_tokenizer.save_model("./bpe")

['./bpe\\vocab.json', './bpe\\merges.txt']

In [8]:
# BERT 토크나이저 구축
wordpiece_tokenizer = BertWordPieceTokenizer(lowercase = False) # lowercase : 소문자변환 여부

In [9]:
wordpiece_tokenizer.train(
    files = ["./data/nsmc/train.txt", "./data/nsmc/test.txt"],
    vocab_size = 10000
)

In [11]:
wordpiece_tokenizer.save_model("./wordpiece")

['./wordpiece\\vocab.txt']

# 모델 입력 데이터 생성

In [59]:
# gpt
tokenizer_gpt = GPT2Tokenizer.from_pretrained("bpe")

In [60]:
tokenizer_gpt.pad_token = "[PAD]"

In [50]:
nsmc.train.get_all_texts()[:3]

('아 더빙.. 진짜 짜증나네요 목소리',
 '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나',
 '너무재밓었다그래서보는것을추천한다')

In [61]:
sentences = [
    '아 더빙.. 진짜 짜증나네요 목소리',
    '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나',
    '너무재밓었다그래서보는것을추천한다'
]

In [62]:
tokenized_sentences = [tokenizer_gpt.tokenize(sentence) for sentence in sentences]

In [63]:
tokenized_sentences

[['ìķĦ', 'ĠëįĶë¹Ļ', '..', 'Ġì§Ħì§ľ', 'Ġì§ľì¦ĿëĤĺ', 'ëĦ¤ìļĶ', 'Ġëª©ìĨĮë¦¬'],
 ['íĿł',
  '...',
  'íı¬ìĬ¤íĦ°',
  'ë³´ê³ł',
  'Ġì´ĪëĶ©',
  'ìĺģíĻĶ',
  'ì¤Ħ',
  '....',
  'ìĺ¤ë²Ħ',
  'ìĹ°ê¸°',
  'ì¡°ì°¨',
  'Ġê°Ģë³į',
  'ì§Ģ',
  'ĠìķĬ',
  'êµ¬ëĤĺ'],
 ['ëĦĪë¬´',
  'ìŀ¬ë',
  '°',
  'ĵ',
  'ìĹĪëĭ¤',
  'ê·¸ëŀĺìĦľ',
  'ë³´ëĬĶ',
  'ê²ĥìĿĦ',
  'ì¶Ķì²ľ',
  'íķľëĭ¤']]

- GPT 모델은 바이트 기준 BPE 를 적용하기 때문에 알 수 없는 문자열로 구성돼있음

In [64]:
batch_inputs = tokenizer_gpt(
    sentences,
    padding = "max_length", # 문장의 최대 길이에 맞춰 패딩
    max_length = 12, # 문장의 토큰 기준 최대 길이
    truncation = True # 문장 잘림 허용
)

In [65]:
batch_inputs

{'input_ids': [[335, 2339, 264, 582, 4056, 465, 3809, 0, 0, 0, 0, 0], [3694, 337, 2877, 759, 2884, 357, 807, 423, 9876, 876, 2961, 7293], [695, 478, 109, 242, 644, 3966, 1068, 2286, 2717, 970, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]]}

- 위 코드의 실행 결과로 2가지 입력값이 만들어짐
    - input_ids : 토큰화 결과를 가지고 각 토큰을 인덱스로 바꾼 것
        - 각 어휘 순서(인덱스)대로 나열된 것
        - 각 토큰을 인덱스로 변환하는 과정을 인덱싱(indexing)이라고 함
        - max_length 에 12를 입력했기 때문에 모든 문장의 길이가 12로 맞춰짐
            - 12보다 짧은 문장은 [PAD] 에 해당하는 인덱스 0이 붙음
        - truncation = True 옵션을 사용하면 원래 토큰 길이가 max_length 보다 길면 길이가 맞춰짐
          
    - attention_mask : 일반 토큰이 자리한 곳과 패딩 토큰이 자리한 곳을 구분

In [31]:
# bert
tokenizer_bert = BertTokenizer.from_pretrained(
    "wordpiece",
    do_lower_case = False
)

In [32]:
tokenized_sentences = [tokenizer_bert.tokenize(sentence) for sentence in sentences]

In [33]:
tokenized_sentences

[['아', '더빙', '.', '.', '진짜', '짜증나', '##네요', '목소리'],
 ['흠',
  '.',
  '.',
  '.',
  '포스터',
  '##보고',
  '초딩',
  '##영화',
  '##줄',
  '.',
  '.',
  '.',
  '.',
  '오버',
  '##연기',
  '##조차',
  '가볍',
  '##지',
  '않',
  '##구나'],
 ['[UNK]']]

In [34]:
batch_inputs = tokenizer_bert(
    sentences,
    padding = "max_length",
    max_length = 12,
    truncation = True
)

In [35]:
batch_inputs

{'input_ids': [[2, 621, 2631, 16, 16, 1993, 3678, 1990, 3323, 3, 0, 0], [2, 997, 16, 16, 16, 2609, 2045, 2796, 1981, 1426, 16, 3], [2, 1, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}

- 코드 실행 결과로 3개의 입력값이 만들어짐
    - input_ids : 토큰 인덱스 시퀀스
        - 각 문장 앞에 2, 끝에 3이 붙어있음
            - 각각 [CLS], [SEP] 토큰에 대응하는 인덱스
            - BERT는 문장 시작과 끝에 이 2개 토큰을 덧붙임
     
    - attention_mask : 일반 토큰이 자리한 곳과 패딩 토큰이 자리한 곳을 구분
 
    - token_type_ids : 세그먼트에 해당하는 것
        - BERT 모델은 기본적으로 문서 혹은 문장 2개를 입력받아서, 둘을 token_type_ids 로 구분
        - 지금은 문장을 하나씩 넣었으므로 모두 0